In [ ]:
using Plots
using ForwardDiff
using QuadGK
using DifferentialEquations
using Apr23Project

# Automatic Differentiation Examples

In [ ]:
f = x-> x^3 + 2x^2 - 5x + 1
df_analytic = x-> 3x^2 + 4x - 5
df_forward_diff = x-> ForwardDiff.derivative(f, x)

In [ ]:
@show df_analytic(1.0);
@show df_forward_diff(1.0);

In [ ]:
@show df_analytic(3.5);
@show df_forward_diff(3.5);

In [ ]:
xx = -5:0.1:5
plot(xx, df_analytic.(xx), label="Analytic Derivative", lw=2)
plot!(xx, df_forward_diff.(xx), label="ForwardDiff Derivative", lw=2, ls=:dash)
xlabel!("x")
ylabel!("f'(x)")
title!("Comparison of Analytic and ForwardDiff Derivatives")

In [ ]:
f = x-> 4 * exp(-x^2) * sin(3x)
df_forward_diff = x-> ForwardDiff.derivative(f, x)

xx = -5:0.1:5
plot(xx, df_forward_diff.(xx), label="ForwardDiff Derivative", lw=2, ls=:dash)
xlabel!("x")
ylabel!("f'(x)")
title!("ForwardDiff Derivatives")

In [ ]:
f = x-> x[1]^2 + x[2]^2
gradf = x-> ForwardDiff.gradient(f, x)

In [ ]:
gradf([1.0, 2.0])

Integrate $y^2$ from 0 to $x$

In [ ]:
g = x-> quadgk(y-> y^2, 0, x)[1]

In [ ]:
@show g(1);

In [ ]:
dg =x -> ForwardDiff.derivative(g, x)

In [ ]:
dg(1.0)

We can't differentiate through `quadgk`

# Integration Packages
`DifferentialEquations.jl`

## Scalar Example
$$
u' = u^2 -t
$$

In [ ]:
u0 = 1.0; #initial condition
tspan = (0.0, 1.0); #time span

# define the ODE problem
prob = ODEProblem(scalar_rhs, u0, tspan)

sol = solve(prob)

We can call `plot` directly on the solution:

In [ ]:
plot(sol, label="ODE Solution", lw=2)

Automatically plots at the `sol.t` and `sol.u`, pairs:

In [ ]:
sol.t

Nonuniformly spaced time steps, by default; this is adaptive time stepping.

In [ ]:
sol.alg

`Tsit5` is the default, adaptive solver

### Running with Explicit and Implicit Euler
And set `adaptive=false` for a fixed time step.

In [ ]:
dt = 0.01
sol_euler = solve(prob, Euler(), dt=dt, adaptive=false)
sol_imp_euler = solve(prob, ImplicitEuler(), dt=dt, adaptive=false)

In [ ]:
sol_imp_euler.t

In [ ]:
plot(sol, label="Automatic ODE Solution", lw=2)
plot!(sol_euler, label="Euler Method", lw=2, ls=:dash)
plot!(sol_imp_euler, label="Implicit Euler Method", lw=2, ls=:dashdot)
xlabel!("Time")
ylabel!("u(t)")
title!("Comparison of ODE Solutions with dt = $(dt)")

Evaluate at fixed points: `t_evals`:

In [ ]:
t_eval = 0:0.2:1;
sol.(t_eval)

In [ ]:
t_eval = 0:0.2:1;
plot(sol, label="ODE Solution", lw=2)
scatter!(t_eval, sol.(t_eval), label="Interpolated Solution", lw=2, ls=:dash)


## Lotka-Volterra Example

In [ ]:
u0 = [1.0, 1.0];
tspan = (0.0, 100.0);
prob_lk = ODEProblem(lk_rhs!, u0, tspan)
# sol_lk = solve(prob_lk)
# use higher precision; more stringent absolute and relative tolerances
sol_lk = solve(prob_lk, abstol=1e-12, reltol=1e-12)

In [ ]:
plot(sol_lk, label="ODE Solution", lw=2, labels=["Prey Population" "Predator Population"])

In comparison to the Euler solution, it maintains the amplitude, and it has closed cycles in the phase plane:

For a vector valud problem, solution is stored as an array of arrays:

In [ ]:
sol_lk.u

In [ ]:
plot([u_[1] for u_ in sol_lk.u], [u_[2] for u_ in sol_lk.u], label="Phase Plot", lw=2)
xlabel!("Prey Population")
ylabel!("Predator Population")
title!("Phase Plot of Lotka-Volterra System")

Note, there is some wobble of the cycles.  We can reduce this by using more stringent error tolerances.